## Imports

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model
import xarray as xr

import numpy as np


## Load Data

In [262]:
is_local_data = False
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3
max_choice = "emp_plus15" #["emp","emp_plus1","emp_plus5","emp_plus10","emp_plus15","Teo_min","Teo_min_plus5","Teo_min_plus10","Teo_min_plus15","mon_plus5","mon_plus10","trash"]

In [263]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

raw_mrsol_for_skew = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=3)

In [264]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")
raw_mrsol_empirical_maximas_skew = raw_mrsol_for_skew.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var,raw_mrsol_empirical_maximas_skew],dim="sets").max("sets")


In [265]:
#for month emp
mon_max_mean = raw_mrsol_for_mean.sel(time=slice(start, end)).resample(time="ME").mean()
mon_max_mean = mon_max_mean.sel(time= mon_max_mean.time.dt.month == Month_idx).max("time")
mon_max_var = raw_mrsol_for_var.sel(time=slice(start, end)).resample(time="ME").mean()
mon_max_var = mon_max_var.sel(time= mon_max_var.time.dt.month == Month_idx).max("time")
mon_max_skew = raw_mrsol_for_skew.sel(time=slice(start, end)).resample(time="ME").mean()
mon_max_skew = mon_max_skew.sel(time= mon_max_skew.time.dt.month == Month_idx).max("time")

raw_mon_mrsol_empirical_maximas = xr.concat([mon_max_mean,mon_max_var,mon_max_skew],dim="sets").max("sets")

In [266]:
maxima_keys = ["emp","emp_plus1","emp_plus5","emp_plus10","emp_plus15","Teo_min","Teo_min_plus5","Teo_min_plus10","Teo_plus15","mon_plus5","mon_plus10","trash"]
max_versions = {}

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = max_v["mrsol"].clip(min=1e-5)
max_versions["emp"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.01).clip(min=1e-5)
max_versions["emp_plus1"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.05).clip(min=1e-5)
max_versions["emp_plus5"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
max_versions["emp_plus10"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.15).clip(min=1e-5)
max_versions["emp_plus15"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = np.maximum(max_v["mrsol"].clip(min=1e-5),(max_v["depth_bnds"].isel(bnds=1)-max_v["depth_bnds"].isel(bnds=0))*1000*0.30)
max_versions["Teo_min"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = np.maximum((max_v["mrsol"]* 1.05).clip(min=1e-5),(max_v["depth_bnds"].isel(bnds=1)-max_v["depth_bnds"].isel(bnds=0))*1000*0.30)
max_versions["Teo_min_plus5"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = np.maximum((max_v["mrsol"]* 1.1).clip(min=1e-5),(max_v["depth_bnds"].isel(bnds=1)-max_v["depth_bnds"].isel(bnds=0))*1000*0.30)
max_versions["Teo_min_plus10"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = np.maximum((max_v["mrsol"]* 1.15).clip(min=1e-5),(max_v["depth_bnds"].isel(bnds=1)-max_v["depth_bnds"].isel(bnds=0))*1000*0.3)
max_versions["Teo_min_plus15"] = max_v

max_v = raw_mon_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.05).clip(min=1e-5)
max_versions["mon_plus5"] = max_v

max_v = raw_mon_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = (max_v["mrsol"] * 1.1).clip(min=1e-5)
max_versions["mon_plus10"] = max_v

max_v = raw_mrsol_empirical_maximas.copy(deep=True)
max_v["mrsol"] = np.maximum((max_v["mrsol"]* 1.15).clip(min=1e-5),(max_v["depth_bnds"].isel(bnds=1)-max_v["depth_bnds"].isel(bnds=0))*100000*0.3)#just too big 
max_versions["trash"] = max_v


approx_maximas = max_versions[max_choice]


In [ ]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

### some functions to shape the data

In [268]:
def shape_target(ds, maximas):
    ds = ds.clip(min = 0)
    ds = (ds/maximas).clip(max = 1-1e-15)
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [269]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [271]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

### Linear Regression of the mean

In [273]:
mean_target_noneT = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target_noneT, chunk_mask, detail_mask = mask_stack_target(mean_target_noneT)
mean_target = model.transform.Logit_Transform_ds(mean_target_noneT)
mean_target_da = mean_target.mrsol

In [274]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [275]:
LinReg_mean = model.stats._parallel_linear_regression.ParLinearRegression()

In [276]:
LinReg_mean.fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

In [ ]:
var_target_noneT = shape_target(raw_mrsol_for_var, approx_maximas)
var_target_noneT, chunk_mask_var, detail_mask_var= mask_stack_target(var_target_noneT)

var_target = model.transform.Logit_Transform_ds(var_target_noneT)



In [278]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [279]:
residuals = LinReg_mean.residuals(var_predictors, var_target)

### Linear Regression of the Variance

In [280]:
LinReg_variance = model.stats._parallel_linear_regression.ParLinearRegression()

In [281]:
LinReg_variance.fit(predictors=var_predictors, target=(residuals.residuals)**2,location_dim="gridcell", regr_dim="time")

### Export Prameters

In [282]:
if safe:
    model.save.save_params(LinReg_mean.params,LinReg_variance.params,maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/Maximas/{max_choice}/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)
